# 01 — CloudWatch Metrics and X-Ray Traces for Bedrock Managed Knowledge Base

Monitor a BMKB using **CloudWatch metrics** in the `AWS/Bedrock/KnowledgeBases` namespace and **X-Ray traces** for per-request latency.

Every KB emits metrics automatically — the KB service role only needs `cloudwatch:PutMetricData` (already added by `ManagedKnowledgeBase`). Traces require an explicit vended-log delivery to X-Ray.

### What this notebook does

1. Creates a small BMKB with an S3 data source
2. Ingests a synthetic document
3. Enables **X-Ray trace delivery** (`TRACES` log type → X-Ray)
4. Generates query traffic (Retrieve) so metrics + traces have data
5. Lists all metrics emitted for this KB and reads counters
6. Reads `RawDataSize` (storage) and `TotalIterationCount` (AgenticRetrieveStream) metrics
7. Reads ingestion application logs
8. Queries X-Ray for per-request latency (this is where latency lives — not CloudWatch)
9. Creates a CloudWatch dashboard bundling the KB's metrics
10. Sets alarms on error rate and throttling
11. Cleans up dashboard, alarms, delivery, and KB

### Prerequisites

- AWS credentials with permissions for Bedrock (`bedrock:*` on knowledge bases), IAM, CloudWatch, CloudWatch Logs, and X-Ray
- **`bedrock:AllowVendedLogDeliveryForResource` on the KB resource** — required by Step 5 to enable trace delivery. If you're using `AdministratorAccess` you already have this; least-privilege setups need it added explicitly. See the [managed-KB observability docs](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-observability.html) for the full policy.
- **X-Ray Transaction Search enabled in this region** — Step 5 (X-Ray trace delivery) will fail unless the account's [X-Ray trace segment destination](https://docs.aws.amazon.com/xray/latest/api/API_UpdateTraceSegmentDestination.html) is set to `CloudWatchLogs` (default is `XRay`). This is an **account+region-wide** change with **cost implications** — it redirects every X-Ray-instrumented service's traces to CloudWatch Logs and bills for ingestion + storage. Step 5 will detect this and abort with clear guidance if you haven't enabled it.
- Enabled model access for the embedding + generation models
- **Kernel:** Select `Python 3`

> ⚠️ **This notebook creates persistent, billable resources** — a KB, S3 bucket, X-Ray delivery pipeline, CloudWatch dashboard, and 2 alarms. **Step 12 has the cleanup, but it is commented out by default so a "Run All" doesn't wipe your KB by accident.** When you're done experimenting, uncomment and run Step 12.

### What Bedrock actually emits

Per the [Managed KB observability docs](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-observability.html):

**Runtime metrics — namespace `AWS/Bedrock/KnowledgeBases`**

| Metric | When | Dimensions |
|--------|------|------------|
| `Invocations` | Every request | `Operation`, `KnowledgeBaseId` (Retrieve only) |
| `ClientErrors` | 4xx (non-throttling) | Same |
| `ServerErrors` | 5xx | Same |
| `Throttles` | 429 | Same |
| `RawDataSize` (Gigabytes) | After ingestion completes | `KnowledgeBaseId` |
| `TotalIterationCount` | AgenticRetrieveStream on success | `Operation` = `AgenticRetrieveStream` |

**No `Latency` metric.** For per-request latency and internal step timing, enable **X-Ray traces**. Traces are emitted for the `Retrieve` operation.

**Ingestion:** per-document status via `APPLICATION_LOGS` delivery — see Step 7.

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --quiet

> ⚠️ **Restart the kernel now, then continue with Step 1.**
>
> The `%pip install` above pulled a newer boto3, but only a fresh kernel picks it up. **Click the restart button (↻) in your notebook toolbar** (Kernel → Restart, or Runtime → Restart on some UIs).
>
> There is no reliable programmatic way to trigger a kernel restart from inside a cell — every documented method either only works in classic Notebook (`Jupyter.notebook.kernel.restart()`) or is a no-op hook (`ip.kernel.do_shutdown()`). The manual button is the harness-agnostic path.

## Step 1 — Configuration

In [ ]:
import boto3
import json
import sys
import time
import logging
import pprint
from datetime import datetime, timedelta, timezone

sys.path.insert(0, '..')

s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
cw = boto3.client('cloudwatch')
logs_client = boto3.client('logs')
xray = boto3.client('xray')

session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration ─────────────────────────────────────────────────────
knowledge_base_name = f'bmkb-obs-{suffix}'
bucket_name = f'bedrock-bmkb-obs-{suffix}-{account_id}'
dashboard_name = f'bmkb-obs-{suffix}'
alarm_name_prefix = f'bmkb-obs-{suffix}'
trace_delivery_source_name = f'bmkb-traces-{suffix}'
trace_delivery_dest_name = f'bmkb-traces-dest-{suffix}'

embedding_model = 'amazon.titan-embed-text-v2:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:     {region}')
print(f'Account:    {account_id}')
print(f'KB Name:    {knowledge_base_name}')
print(f'Bucket:     {bucket_name}')
print(f'Dashboard:  {dashboard_name}')

## Step 2 — Create S3 bucket and upload a document

In [ ]:
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Bucket {bucket_name} already exists')
except Exception:
    print(f'Creating bucket {bucket_name}')
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region},
        )

file_to_upload = '../synthetic_dataset/octank_financial_10K.pdf'
print(f'Uploading {file_to_upload} to s3://{bucket_name}/')
s3_client.upload_file(file_to_upload, bucket_name, 'octank_financial_10K.pdf')
print('Done.')

## Step 3 — Create the Bedrock Managed Knowledge Base

`enable_logging=True` provisions the CloudWatch Logs delivery for `APPLICATION_LOGS` (ingestion status). We add the X-Ray traces delivery separately in the next step.

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    bucket_name=bucket_name,
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')
print(f'DS ID: {kb.ds_id}')

kb_arn = f'arn:aws:bedrock:{region}:{account_id}:knowledge-base/{kb.kb_id}'
kb_id = kb.kb_id
%store kb_id

## Step 4 — Ingest documents

Once the ingestion job completes, Bedrock publishes the `RawDataSize` metric.

In [ ]:
job = kb.start_ingestion_job()

## Step 5 — Enable X-Ray trace delivery for Retrieve

Bedrock delivers per-request Retrieve traces to X-Ray via the vended-log delivery pipeline: `PutDeliverySource(logType=TRACES)` → `PutDeliveryDestination(type=XRAY)` → `CreateDelivery`. All three calls are idempotent — running this cell twice is safe.

X-Ray is a managed destination — you don't specify a `destinationResourceArn` for the destination.

### Preflight — X-Ray Transaction Search must be enabled

Bedrock's `X-Ray` delivery destination only works when the account's X-Ray **trace segment destination** is `CloudWatchLogs` (i.e., X-Ray Transaction Search is on). The default is `XRay`, in which case `CreateDelivery` fails with:

> `X-Ray Delivery Destination is supported with CloudWatch Logs as a Trace Segment Destination. Please enable ... using the UpdateTraceSegmentDestination API`

The preflight cell **checks** the current setting and stops if it's not enabled. It does **not** flip the setting for you — enabling Transaction Search is an **account+region-wide** change that redirects every X-Ray-instrumented service's traces to CloudWatch Logs and bills for ingestion + storage. Confirm with your account owner before enabling.

In [ ]:
# Preflight: check X-Ray trace segment destination
tsd = xray.get_trace_segment_destination()
current = tsd.get('Destination')
status = tsd.get('Status')
print(f'X-Ray trace segment destination: {current} ({status})')

if current != 'CloudWatchLogs':
    print()
    print('❌ Cannot enable Bedrock KB → X-Ray trace delivery on this account/region.')
    print('   The trace segment destination is currently "XRay" (default).')
    print('   Bedrock requires it to be "CloudWatchLogs" (X-Ray Transaction Search).')
    print()
    print('   To enable — this is an ACCOUNT+REGION-WIDE change with cost implications:')
    print('     xray.update_trace_segment_destination(Destination="CloudWatchLogs")')
    print()
    print('   Skip this step (and Step 9 X-Ray traces) if you cannot make that change.')
    raise SystemExit('Preflight failed — see message above')

In [ ]:
def _idempotent(fn, label, expected=('conflict','already exists','already_exists')):
    try:
        r = fn()
        print(f'  ✓ {label}')
        return r
    except Exception as e:
        msg = str(e).lower()
        if any(k in msg for k in expected):
            print(f'  ✓ {label} (already exists)')
            return None
        raise

# 1. Delivery source — the KB, log type TRACES
_idempotent(
    lambda: logs_client.put_delivery_source(
        name=trace_delivery_source_name,
        resourceArn=kb_arn,
        logType='TRACES',
    ),
    f'PutDeliverySource {trace_delivery_source_name!r}',
)

# 2. Delivery destination — X-Ray (no destinationResourceArn for XRAY)
dest_resp = _idempotent(
    lambda: logs_client.put_delivery_destination(
        name=trace_delivery_dest_name,
        deliveryDestinationType='XRAY',
    ),
    f'PutDeliveryDestination {trace_delivery_dest_name!r} (XRAY)',
)

# For the CreateDelivery call, we need the destination ARN. If it already
# existed, describe it to get the ARN.
if dest_resp:
    dest_arn = dest_resp['deliveryDestination']['arn']
else:
    dest_arn = logs_client.get_delivery_destination(name=trace_delivery_dest_name)['deliveryDestination']['arn']

# 3. Delivery — link source → destination
_idempotent(
    lambda: logs_client.create_delivery(
        deliverySourceName=trace_delivery_source_name,
        deliveryDestinationArn=dest_arn,
    ),
    'CreateDelivery',
)
print(f'\nX-Ray trace delivery enabled for {kb_arn}')

## Step 6 — Generate query traffic

Metrics and traces don't appear until the KB is invoked. We fire several Retrieve calls, then wait 90 seconds for CloudWatch metric aggregation.

In [ ]:
queries = [
    'What is Octank Financial\'s total revenue?',
    'Summarize the risk factors in the annual report.',
    'What are the key business segments?',
    'Describe the executive compensation structure.',
    'What was the net income for the fiscal year?',
]

print('=== Retrieve traffic ===')
# kb.retrieve() currently omits retrievalConfiguration due to an SDK/service
# shape mismatch (see the utility docstring). The service applies its own
# default (~5 chunks) — so the reported N is what Bedrock returned, not
# something we set here.
for q in queries:
    try:
        resp = kb.retrieve(q)
        n = len(resp.get('retrievalResults', []))
        print(f'  retrieve  "{q[:60]}..." -> {n} chunks')
    except Exception as e:
        print(f'  retrieve raised: {type(e).__name__}: {str(e)[:120]}')

print('\nWaiting 90s for CloudWatch metrics propagation...')
time.sleep(90)

## Step 7 — List and read metrics

`list_metrics` returns one entry per unique (metric-name, dimension-set). Grouped by operation for readability.

In [ ]:
kb_dimension = f'knowledge-base/{kb.kb_id}'

available = cw.list_metrics(
    Namespace='AWS/Bedrock/KnowledgeBases',
    Dimensions=[{'Name': 'KnowledgeBaseId', 'Value': kb_dimension}],
)

print(f'=== {len(available["Metrics"])} unique (metric, dims) tuples for {kb_dimension} ===\n')

by_op = {}
for m in available['Metrics']:
    dims = {d['Name']: d['Value'] for d in m['Dimensions']}
    op = dims.get('Operation', '(no-operation)')
    by_op.setdefault(op, []).append(m['MetricName'])

for op, names in sorted(by_op.items()):
    print(f'  Operation={op}')
    for n in sorted(names):
        print(f'    - {n}')

### 7a. Runtime counters (Invocations, ClientErrors, ServerErrors, Throttles)

Only `Invocations` publishes for every request. The other three are conditional — you'll see `no-emit` under normal conditions, which is what you want.

In [ ]:
# datetime.utcnow() is deprecated in Python 3.12+; use tz-aware datetimes.
# boto3 accepts either.
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=1)

COUNTER_METRICS = ['Invocations', 'ClientErrors', 'ServerErrors', 'Throttles']

def _print_counter_totals(operation):
    print(f'\n=== Operation: {operation} (last hour) ===')
    for name in COUNTER_METRICS:
        resp = cw.get_metric_statistics(
            Namespace='AWS/Bedrock/KnowledgeBases',
            MetricName=name,
            Dimensions=[
                {'Name': 'KnowledgeBaseId', 'Value': kb_dimension},
                {'Name': 'Operation', 'Value': operation},
            ],
            StartTime=start_time,
            EndTime=end_time,
            Period=300,
            Statistics=['Sum'],
        )
        total = int(sum(dp['Sum'] for dp in resp.get('Datapoints', [])))
        emitted = 'emitted' if total > 0 else 'no-emit'
        print(f'  {name:15s}: {total:>4d}  ({emitted})')

_print_counter_totals('Retrieve')

### 7b. Storage metric (`RawDataSize`)

Published once per completed ingestion job — not per API call. Dimension is only `KnowledgeBaseId` (no `Operation`).

In [ ]:
resp = cw.get_metric_statistics(
    Namespace='AWS/Bedrock/KnowledgeBases',
    MetricName='RawDataSize',
    Dimensions=[{'Name': 'KnowledgeBaseId', 'Value': kb_dimension}],
    StartTime=start_time,
    EndTime=end_time,
    Period=3600,
    Statistics=['Maximum'],
)
datapoints = resp.get('Datapoints', [])
if datapoints:
    latest = sorted(datapoints, key=lambda x: x['Timestamp'])[-1]
    print(f'RawDataSize: {latest["Maximum"]:.4f} GB (as of {latest["Timestamp"]})')
else:
    print('RawDataSize: no data yet — this metric is published after ingestion completes; wait a few minutes.')

## Step 8 — Read ingestion application logs

The ingestion pipeline emits per-document status events to CloudWatch Logs. Two `event_type` values:

- `StartIngestionJob.StatusChanged` — job-level (crawling started, crawling completed, etc.)
- `StartIngestionJob.ResourceStatusChanged` — per-document (with `crawl_status` / `sync_status` / `index_status`)

In [ ]:
kb_log_group = f'/aws/vendedlogs/bedrock/knowledge-base/APPLICATION_LOGS/{kb.kb_id}'

try:
    end_ms = int(time.time() * 1000)
    start_ms = end_ms - (6 * 60 * 60 * 1000)   # last 6 hours
    events = logs_client.filter_log_events(
        logGroupName=kb_log_group,
        startTime=start_ms,
        endTime=end_ms,
        limit=20,
    ).get('events', [])
    print(f'=== KB Application Logs ({len(events)} events, last 6h) ===\n')
    for e in events[:20]:
        try:
            log = json.loads(e['message'])
            event_type = log.get('event_type', '?')
            ev = log.get('event', {})
            if event_type == 'StartIngestionJob.StatusChanged':
                print(f'  [job]    {ev.get("message", "")}')
            else:
                doc = ev.get('source_uri', ev.get('document_id', '?'))
                consolidated = ev.get('connector_document_status', {}).get('Status', '?')
                index = ev.get('index_status', {}).get('Status', '?')
                sync = ev.get('sync_status', {}).get('Status', '?')
                crawl = ev.get('crawl_status', {}).get('Status', '?')
                err = ev.get('error_message')
                print(f'  [doc]    {doc}: crawl={crawl}, sync={sync}, index={index}, overall={consolidated}')
                if err:
                    print(f'           error: {err}')
        except json.JSONDecodeError:
            print(f'  (raw) {e["message"][:200]}')
except logs_client.exceptions.ResourceNotFoundException:
    print(f'Log group not found: {kb_log_group}')
    print('The log group is created by enable_logging=True on ManagedKnowledgeBase.')

## Step 9 — Query X-Ray for per-request latency

**This is where latency lives** — not CloudWatch. Bedrock delivers a Retrieve trace per request when you enable the `TRACES` log type (Step 5).

Traces surface within ~1 minute of a request. Below we list recent trace summaries for our KB and read the fastest / slowest.

In [ ]:
# X-Ray trace summaries by service filter (KB ARN in origin, operation="Retrieve")
xray_end = datetime.now(timezone.utc)
xray_start = xray_end - timedelta(minutes=15)

# Filter: annotations set by Bedrock make service filtering possible; broad filter as fallback
resp = xray.get_trace_summaries(
    StartTime=xray_start,
    EndTime=xray_end,
    FilterExpression=f'service(id(name: "{knowledge_base_name}", type: "AWS::Bedrock::KnowledgeBase")) OR annotation.knowledgeBaseId = "{kb.kb_id}"',
    Sampling=False,
)
summaries = resp.get('TraceSummaries', [])

if not summaries:
    # Broad fallback — most recent traces regardless of service
    resp = xray.get_trace_summaries(StartTime=xray_start, EndTime=xray_end, Sampling=False)
    summaries = [
        s for s in resp.get('TraceSummaries', [])
        if any(kb.kb_id in (svc.get('Names') or []) or kb.kb_id in json.dumps(svc, default=str) for svc in s.get('ServiceIds', []))
    ]

print(f'=== {len(summaries)} recent Retrieve traces (last 15 min) ===')
if not summaries:
    print('  No traces yet. Traces surface within ~1 min of the Retrieve call — retry this cell.')
else:
    durations = sorted(s['Duration'] * 1000 for s in summaries if s.get('Duration'))
    print(f'  count={len(durations)}, min={durations[0]:.1f}ms, max={durations[-1]:.1f}ms')
    if len(durations) >= 3:
        p50 = durations[len(durations)//2]
        p90_i = int(len(durations) * 0.9)
        print(f'  p50={p50:.1f}ms, p90={durations[p90_i]:.1f}ms')

    print(f'\n  === First trace detail ===')
    trace_id = summaries[0]['Id']
    # X-Ray is eventually consistent — a summary can appear before the full
    # trace document is queryable, so batch_get_traces can return Traces=[].
    traces = xray.batch_get_traces(TraceIds=[trace_id]).get('Traces', [])
    if not traces:
        print(f'    (trace {trace_id} not yet retrievable — X-Ray is eventually consistent, retry in ~30s)')
    else:
        for seg_str in traces[0]['Segments'][:3]:
            seg = json.loads(seg_str['Document'])
            name = seg.get('name', '?')
            start = seg.get('start_time', 0)
            end = seg.get('end_time', 0)
            print(f'    segment {name}: {(end-start)*1000:.1f}ms')

## Step 10 — Create a CloudWatch dashboard

Bundle the KB's counter metrics into one dashboard. Latency panels here would be dead — use the X-Ray console for latency, or embed a trace map widget.

In [ ]:
def _widget(title, metric, x, y, ops=('Retrieve',)):
    metrics_arr = [
        ['AWS/Bedrock/KnowledgeBases', metric,
         'KnowledgeBaseId', kb_dimension,
         'Operation', op,
         {'stat': 'Sum', 'label': f'{metric} {op}'}]
        for op in ops
    ]
    return {
        'type': 'metric',
        'x': x, 'y': y, 'width': 8, 'height': 6,
        'properties': {
            'metrics': metrics_arr,
            'view': 'timeSeries',
            'stacked': False,
            'region': region,
            'title': title,
            'period': 300,
        },
    }

def _text_widget(text, x, y, width=24, height=2):
    return {
        'type': 'text',
        'x': x, 'y': y, 'width': width, 'height': height,
        'properties': {'markdown': text},
    }

dashboard_body = {
    'widgets': [
        _widget('Invocations',   'Invocations',   x=0,  y=0),
        _widget('ClientErrors',  'ClientErrors',  x=8,  y=0),
        _widget('ServerErrors',  'ServerErrors',  x=16, y=0),
        _widget('Throttles',     'Throttles',     x=0,  y=6),
        _text_widget(
            f'**For per-request latency, see the [X-Ray console]'
            f'(https://{region}.console.aws.amazon.com/xray/home?region={region}#/traces).**  \n'
            f'Traces are delivered via the TRACES log type set up in the notebook.',
            x=8, y=6, width=16, height=6,
        ),
    ]
}

cw.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body),
)
print(f'Created dashboard: {dashboard_name}')
print(f'View: https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}')

## Step 11 — Set CloudWatch alarms

Two demo alarms. Neither has an SNS action wired up — add `AlarmActions=[sns_topic_arn]` for real paging.

In [ ]:
error_alarm = f'{alarm_name_prefix}-retrieve-server-errors'
cw.put_metric_alarm(
    AlarmName=error_alarm,
    AlarmDescription='BMKB Retrieve ServerErrors > 0 in a 5-min period',
    Namespace='AWS/Bedrock/KnowledgeBases',
    MetricName='ServerErrors',
    Dimensions=[
        {'Name': 'KnowledgeBaseId', 'Value': kb_dimension},
        {'Name': 'Operation', 'Value': 'Retrieve'},
    ],
    Statistic='Sum',
    Period=300,
    EvaluationPeriods=1,
    Threshold=0,
    ComparisonOperator='GreaterThanThreshold',
    TreatMissingData='notBreaching',
    ActionsEnabled=False,
)
print(f'Created alarm: {error_alarm}')

throttle_alarm = f'{alarm_name_prefix}-retrieve-throttles'
cw.put_metric_alarm(
    AlarmName=throttle_alarm,
    AlarmDescription='BMKB Retrieve Throttles > 0 in a 5-min period',
    Namespace='AWS/Bedrock/KnowledgeBases',
    MetricName='Throttles',
    Dimensions=[
        {'Name': 'KnowledgeBaseId', 'Value': kb_dimension},
        {'Name': 'Operation', 'Value': 'Retrieve'},
    ],
    Statistic='Sum',
    Period=300,
    EvaluationPeriods=1,
    Threshold=0,
    ComparisonOperator='GreaterThanThreshold',
    TreatMissingData='notBreaching',
    ActionsEnabled=False,
)
print(f'Created alarm: {throttle_alarm}')

resp = cw.describe_alarms(AlarmNamePrefix=alarm_name_prefix)
print(f'\n=== Alarms with prefix {alarm_name_prefix!r} ===')
for a in resp.get('MetricAlarms', []) + resp.get('CompositeAlarms', []):
    print(f'  {a["AlarmName"]}: {a["StateValue"]}')

## Step 12 — Cleanup

> Uncomment the block below to delete everything. Delete order: alarms → dashboard → trace delivery → KB → S3 objects → bucket.

In [ ]:
# print('Deleting alarms and dashboard...')
# cw.delete_alarms(AlarmNames=[error_alarm, throttle_alarm])
# cw.delete_dashboards(DashboardNames=[dashboard_name])
#
# print('Deleting X-Ray trace delivery...')
# # Paginate: describe_deliveries returns at most 100 per page. Filtering on
# # the caller side is required — the API takes no filters. Deliveries must
# # be deleted BEFORE sources/destinations, otherwise the delete raises
# # ConflictException with 'currently in use'.
# paginator = logs_client.get_paginator('describe_deliveries')
# for page in paginator.paginate():
#     for delivery in page.get('deliveries', []):
#         if delivery.get('deliverySourceName') == trace_delivery_source_name:
#             logs_client.delete_delivery(id=delivery['id'])
# try:
#     logs_client.delete_delivery_source(name=trace_delivery_source_name)
# except Exception:
#     pass
# try:
#     logs_client.delete_delivery_destination(name=trace_delivery_dest_name)
# except Exception:
#     pass
#
# print('Deleting KB, IAM role, and CloudWatch log delivery...')
# kb.delete_kb(delete_iam=True)
#
# print(f'Emptying and deleting bucket {bucket_name}...')
# resource = boto3.resource('s3')
# resource.Bucket(bucket_name).objects.all().delete()
# resource.Bucket(bucket_name).delete()
# print('Done.')

## Summary

| Signal | Where | How to read it |
|--------|-------|----------------|
| Invocations, Errors, Throttles | `AWS/Bedrock/KnowledgeBases` metrics | `cloudwatch.get_metric_statistics(Statistics=['Sum'])` |
| Storage size (`RawDataSize`, GB) | Same namespace, no `Operation` dim | Same, use `Statistics=['Maximum']` |
| Ingestion status per document | `/aws/vendedlogs/bedrock/knowledge-base/APPLICATION_LOGS/{kb_id}` | `logs.filter_log_events` |
| Per-request latency (Retrieve) | X-Ray traces | `xray.get_trace_summaries` |

### Dimensions

- `KnowledgeBaseId` — always in the form `knowledge-base/{kb_id}` (with prefix). Present on `Retrieve` metrics and `RawDataSize`.
- `Operation` — `Retrieve` or `AgenticRetrieveStream`. Some operations publish only with the `Operation` dimension.

### Recommended alarms

| Alarm | Metric | Threshold | Notes |
|-------|--------|-----------|-------|
| Retrieve error rate | `ServerErrors` (Sum) on `Retrieve` | > 0 in 5min | Any 5xx is a page |
| Retrieve throttling | `Throttles` (Sum) on `Retrieve` | > 0 in 5min | Increase quota or add retries |

For latency alarms, use CloudWatch Alarms on **X-Ray insight metrics** or a custom metric derived from the trace stream. See [X-Ray insights](https://docs.aws.amazon.com/xray/latest/devguide/xray-console-insights.html).

### Console shortcuts

- **CloudWatch dashboards:** https://console.aws.amazon.com/cloudwatch/home#dashboards
- **X-Ray traces:** https://console.aws.amazon.com/xray/home
- **Bedrock console → Knowledge Bases → your KB → Monitoring** — pre-built view over the same data